<a href="https://colab.research.google.com/github/Ilya-creat/python-group-progject2/blob/colab/gp2_analyst_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Программирование на Python, БИ

## НИУ ВШЭ, 2025-26 учебный год

# Групповой проект №2. DE: Scraping & API

### Общая информация

Дата выдачи: 21.10.2025

Защита проекта: 12.11.2025

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import plotly.express as px
import numpy as np
df = pd.read_csv('sstats_data_normal-20251104_100000.csv')
len(df)

In [ ]:
for c in ['Match Winner_Home','Match Winner_Draw','Match Winner_Away']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['odds_std'] = df[['Match Winner_Home','Match Winner_Draw','Match Winner_Away']].std(axis=1)
df['odds_var'] = df[['Match Winner_Home','Match Winner_Draw','Match Winner_Away']].var(axis=1)

print(df[['Match Winner_Home','Match Winner_Draw','Match Winner_Away','odds_std','odds_var']].head())

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df['odds_std'], bins=30, kde=True)
plt.title('Распределение стандартного отклонения коэффициентов в матче')
plt.xlabel('Стандартное отклонение')
max_x = df['odds_std'].quantile(0.98)
plt.xlim(0, max_x)
plt.ylabel('Частота')
plt.show()


In [ ]:
for col in ['Match Winner_Home', 'Match Winner_Draw', 'Match Winner_Away']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

mask = ((df['Match Winner_Home'].between(2.5, 3.5)) &(df['Match Winner_Draw'].between(2.5, 3.5)) &(df['Match Winner_Away'].between(2.5, 3.5)))

share_equal_odds = mask.mean()
print(f'Доля матчей с примерно равными шансами: {share_equal_odds:.4%}')

In [ ]:
for col in ['Match Winner_Home', 'Match Winner_Draw', 'Match Winner_Away']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['odds_range'] = df[['Match Winner_Home', 'Match Winner_Draw', 'Match Winner_Away']].max(axis=1) - \
df[['Match Winner_Home', 'Match Winner_Draw', 'Match Winner_Away']].min(axis=1)

share_no = (df['odds_range'] <= 0.5).mean()
print(f'Доля матчей без явного фаворита: {share_no:.2%}')


In [ ]:
for c in ['Match Winner_Home', 'Match Winner_Draw', 'Match Winner_Away']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['outcome'] = pd.Series(
    ['Home' if h > a else 'Away' if h < a else 'Draw'
     for h, a in zip(df['homeResult'], df['awayResult'])]
)

df['fav'] = df[['Match Winner_Home','Match Winner_Draw','Match Winner_Away']].idxmin(axis=1)
df['dog'] = df[['Match Winner_Home','Match Winner_Draw','Match Winner_Away']].idxmax(axis=1)

df['fav_won'] = (
    (df['fav'].eq('Match Winner_Home') & df['outcome'].eq('Home')) |
    (df['fav'].eq('Match Winner_Away') & df['outcome'].eq('Away')) |
    (df['fav'].eq('Match Winner_Draw') & df['outcome'].eq('Draw'))
)
shock_share = (~df['fav_won']).mean()
print(f'Доля неожиданных результатов (фаворит не выиграл): {shock_share:.2%}')

league_shocks = df.groupby('leagueName')['fav_won'].apply(lambda x: 1 - x.mean()).sort_values(ascending=False)
print('\nТОП лиг по доле неожиданных результатов:')
print(league_shocks.head(10))

bk_shocks = df.groupby('bookmakerName')['fav_won'].apply(lambda x: 1 - x.mean()).sort_values(ascending=False)
print('\nТОП букмекеров по доле неожиданных результатов:')
print(bk_shocks.head(10))

In [ ]:
cols = ['Match Winner_Home','Match Winner_Draw','Match Winner_Away']
for c in cols: df[c] = pd.to_numeric(df[c], errors='coerce')
df['homeResult'] = pd.to_numeric(df['homeResult'], errors='coerce')
df['awayResult'] = pd.to_numeric(df['awayResult'], errors='coerce')
df['odds_std'] = df[cols].std(axis=1)

df['fav'] = df[cols].idxmin(axis=1)
df['dog'] = df[cols].idxmax(axis=1)

df['winner_col'] = np.select(
    [df['homeResult']>df['awayResult'], df['homeResult']==df['awayResult'], df['homeResult']<df['awayResult']],
    ['Match Winner_Home','Match Winner_Draw','Match Winner_Away'],
    default='Unknown'
)

df['unpredictability'] = np.where(df['winner_col'].eq(df['dog']), 1,
                           np.where(df['winner_col'].eq(df['fav']), 0, 0.5))

plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='odds_std', y='unpredictability', hue='bookmakerName', s=25, alpha=0.6)
mx = df['odds_std'].quantile(0.99)
if pd.notna(mx): plt.xlim(0, mx)
plt.title('Дисперсия коэффициентов vs фактическая непредсказуемость')
plt.xlabel('Стандартное отклонение коэффициентов (по матчу)')
plt.ylabel('Непредсказуемость: 0 (фаворит выиграл) – 1 (аутсайдер выиграл)')
plt.legend(loc='best', ncol=2, fontsize=8, title='bookmaker')
plt.show()


In [ ]:
for c in ['Match Winner_Home','Match Winner_Draw','Match Winner_Away']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

top_leagues = df['leagueName'].value_counts().head(10).index
d = df[df['leagueName'].isin(top_leagues)]

plt.figure(figsize=(14,6))
sns.violinplot(data=d, x='leagueName', y='Match Winner_Draw', inner='quartile', cut=0)
plt.xticks(rotation=60)
plt.xlabel('Лига'); plt.ylabel('Коэф. на ничью'); plt.title('Match Winner_Draw по лигам (топ-10)')
plt.show()

df['odds_range'] = df[['Match Winner_Home','Match Winner_Draw','Match Winner_Away']].max(axis=1) - \
                   df[['Match Winner_Home','Match Winner_Draw','Match Winner_Away']].min(axis=1)
df['strength'] = pd.cut(df['odds_range'], bins=[-np.inf,0.5,1.5,np.inf],
                        labels=['Равные (≤0.5)','Средний (0.5–1.5)','Фаворит (>1.5)'])

plt.figure(figsize=(10,6))
sns.violinplot(data=df, x='strength', y='Match Winner_Draw', inner='quartile', cut=0)
plt.xlabel('Группа матча');
plt.ylabel('Коэф. на ничью');
plt.title('Match Winner_Draw по «силе» матча')
plt.show()


In [ ]:
cols = ['Match Winner_Home','Match Winner_Draw','Match Winner_Away']
for c in cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

std_df = ( df.groupby(['leagueName','bookmakerName'])[cols].std().mean(axis=1) .reset_index(name='std_mean'))

top_leagues = df['leagueName'].value_counts().head(5).index
top_books   = df['bookmakerName'].value_counts().head(5).index

plt.figure(figsize=(10,6))
sns.boxplot(data=std_df[std_df['leagueName'].isin(top_leagues)],
            x='leagueName', y='std_mean')
plt.title('Дисперсия коэффициентов — топ-5 популярных лиг')
plt.xlabel('Лига'); plt.ylabel('Среднее стандартное отклонение')
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(10,6))
sns.boxplot(data=std_df[std_df['bookmakerName'].isin(top_books)],
            x='bookmakerName', y='std_mean')
plt.title('Дисперсия коэффициентов — топ-5 популярных букмекеров')
plt.xlabel('Букмекер'); plt.ylabel('Среднее стандартное отклонение')
plt.xticks(rotation=45)
plt.show()


In [ ]:
for c in ['Match Winner_Home', 'Match Winner_Draw', 'Match Winner_Away']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['implied_home'] = 1 / df['Match Winner_Home']
df['implied_draw'] = 1 / df['Match Winner_Draw']
df['implied_away'] = 1 / df['Match Winner_Away']

df['home_win'] = (df['homeResult'] > df['awayResult']).astype(int)
df['draw']     = (df['homeResult'] == df['awayResult']).astype(int)
df['away_win'] = (df['homeResult'] < df['awayResult']).astype(int)

df['err_home'] = abs(df['implied_home'] - df['home_win'])
df['err_draw'] = abs(df['implied_draw'] - df['draw'])
df['err_away'] = abs(df['implied_away'] - df['away_win'])
df['err_mean'] = df[['err_home','err_draw','err_away']].mean(axis=1)

plt.figure(figsize=(14,6))
sns.boxplot(data=df, x='leagueName', y='err_mean')
plt.title('Ошибка букмекера по лигам')
plt.xlabel('Лига'); plt.ylabel('Средняя ошибка')
plt.xticks(rotation=60)
plt.show()

plt.figure(figsize=(12,6))
sns.boxplot(data=df, x='bookmakerName', y='err_mean')
plt.title('Ошибка букмекера по букмекерам')
plt.xlabel('Букмекер'); plt.ylabel('Средняя ошибка')
plt.xticks(rotation=45)
plt.show()


In [ ]:
top_leagues = df['leagueName'].value_counts().head(20).index
df_top = df[df['leagueName'].isin(top_leagues)]

plt.figure(figsize=(12,6))
sns.boxplot(data=df_top, x='leagueName', y='err_mean')
plt.title('Ошибка букмекера по топ-20 лигам')
plt.xlabel('Лига')
plt.ylabel('Средняя ошибка')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
cols = ['Match Winner_Home','Match Winner_Draw','Match Winner_Away']
for c in cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['bookmaker_margin'] = (1/df['Match Winner_Home'] +1/df['Match Winner_Draw'] + 1/df['Match Winner_Away'] - 1)

tmp = (df.replace([np.inf, -np.inf], np.nan).dropna(subset=['bookmaker_margin','leagueName']))

league_margin = (tmp.groupby('leagueName')['bookmaker_margin']
                   .mean()
                   .sort_values(ascending=False)
                   .head(20))

plt.figure(figsize=(12,6))
sns.barplot(x=league_margin.values, y=league_margin.index)
plt.title('Средняя маржа букмекеров по лигам (топ-20)')
plt.xlabel('Средняя маржа'); plt.ylabel('Лига')
plt.tight_layout()
plt.show()


In [ ]:
df['implied_home'] = 1 / pd.to_numeric(df['Match Winner_Home'], errors='coerce')
df['implied_away'] = 1 / pd.to_numeric(df['Match Winner_Away'], errors='coerce')
df['implied_draw'] = 1 / pd.to_numeric(df['Match Winner_Draw'], errors='coerce')

df['actual_home'] = (df['homeResult'] > df['awayResult']).astype(int)
df['actual_draw'] = (df['homeResult'] == df['awayResult']).astype(int)
df['actual_away'] = (df['homeResult'] < df['awayResult']).astype(int)

df['bookmaker_error'] = (    abs(df['implied_home'] - df['actual_home']) +    abs(df['implied_draw'] - df['actual_draw']) +    abs(df['implied_away'] - df['actual_away'])) / 3

heat = (
    df.groupby(['bookmakerName', 'leagueName'])['bookmaker_error'].mean().unstack()
)

plt.figure(figsize=(14,6))
sns.heatmap(heat, cmap='coolwarm', linewidths=0.5)
plt.title('Средняя ошибка букмекеров по лигам')
plt.xlabel('Лига')
plt.ylabel('Букмекер')
plt.tight_layout()
plt.show()


In [ ]:
top_leagues = df['leagueName'].value_counts().head(5).index
low_leagues = df['leagueName'].value_counts().tail(5).index

df_top = df[df['leagueName'].isin(top_leagues)]
df_low = df[df['leagueName'].isin(low_leagues)]

plt.figure(figsize=(12,6))
sns.histplot(df_top['Match Winner_Home'], kde=True, color='blue', label='Популярные лиги', stat='density')
sns.histplot(df_low['Match Winner_Home'], kde=True, color='red', label='Непопулярные лиги', stat='density', alpha=0.6)

plt.xlim(1,10)
plt.title('Распределение коэффициентов на победу хозяев\n(Популярные vs Непопулярные лиги)')
plt.xlabel('Коэффициент')
plt.ylabel('Плотность')
plt.legend()
plt.show()


In [ ]:
top_rounds = df['roundName'].value_counts().head(15).index
df_rounds = df[df['roundName'].isin(top_rounds)]

round_diff = (df_rounds.groupby('roundName').agg({
          'actual_home': 'mean',
          'actual_draw': 'mean',
          'actual_away': 'mean',
          'implied_home': 'mean',
          'implied_draw': 'mean',
          'implied_away': 'mean'
      })
)
round_diff['diff_home'] = round_diff['actual_home'] - round_diff['implied_home']
round_diff['diff_draw'] = round_diff['actual_draw'] - round_diff['implied_draw']
round_diff['diff_away'] = round_diff['actual_away'] - round_diff['implied_away']

plt.figure(figsize=(12,6))
sns.barplot(data=round_diff[['diff_home','diff_draw','diff_away']].reset_index().melt(id_vars='roundName'),
    x='roundName', y='value', hue='variable')
plt.title('Разница между фактической и подразумеваемой вероятностью (топ-15 стадий)')
plt.xlabel('Стадия турнира')
plt.ylabel('Средняя разница (actual - implied)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Исход')
plt.tight_layout()
plt.show()


In [ ]:
round_diff = (df.groupby('roundName').agg({
          'actual_home': 'mean',
          'actual_draw': 'mean',
          'actual_away': 'mean',
          'implied_home': 'mean',
          'implied_draw': 'mean',
          'implied_away': 'mean'
      })
)

round_diff['diff_home'] = round_diff['actual_home'] - round_diff['implied_home']
round_diff['diff_draw'] = round_diff['actual_draw'] - round_diff['implied_draw']
round_diff['diff_away'] = round_diff['actual_away'] - round_diff['implied_away']

round_diff['max_abs_diff'] = round_diff[['diff_home','diff_draw','diff_away']].abs().max(axis=1)
top_problem_rounds = round_diff.sort_values('max_abs_diff', ascending=False).head(10)

print("🔍 ТОП-10 стадий турнира с наибольшими расхождениями (нишевые проблемы БК):")
print(top_problem_rounds[['diff_home','diff_draw','diff_away','max_abs_diff']].round(3))


In [ ]:
round_diff = (
    df.groupby('roundName')[['actual_home', 'implied_home','actual_draw', 'implied_draw','actual_away', 'implied_away']]
    .mean()
)

round_diff['diff_home'] = round_diff['actual_home'] - round_diff['implied_home']
round_diff['diff_draw'] = round_diff['actual_draw'] - round_diff['implied_draw']
round_diff['diff_away'] = round_diff['actual_away'] - round_diff['implied_away']


plt.figure(figsize=(12,6))
round_diff[['diff_home','diff_draw','diff_away']].plot(kind='bar')
plt.title('Средняя разница (actual - implied) по стадиям турнира')
plt.xlabel('Стадия турнира')
plt.ylabel('Разница между фактом и прогнозом')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
df = pd.read_csv('bk_offers_hist_normal-20251106_160848.csv')
len(df)

In [ ]:
market_counts = df['market'].value_counts(dropna=False)

market_percent = (market_counts / len(df) * 100).round(2)

print("📊 Частота типов рынков:")
print(market_counts)
print("\n📈 Процентное соотношение рынков:")
print(market_percent)

plt.figure(figsize=(6,4))
market_percent.plot(kind='bar', color=['skyblue','orange','lightgreen'])
plt.title('Доля типов рынков в датасете')
plt.ylabel('Процент от общего числа записей')
plt.xticks(rotation=0)
plt.show()


In [ ]:
market_counts = df['market'].value_counts().reset_index()
market_counts.columns = ['market','count']

fig = px.pie(
    market_counts,
    names='market',
    values='count',
    title='Распределение типов рынков в датасете',
    hole=0.35,
    color='market',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(textposition='inside', textinfo='label+percent', hovertemplate='%{label}: %{value} записей (%{percent})')
fig.update_layout(
    template='plotly_white',
    legend_title_text='market',
    uniformtext_minsize=12,
    uniformtext_mode='hide',
    height=500
)

fig.show()



In [ ]:
book_col = 'bookmaker_title' if 'bookmaker_title' in df.columns else 'bookmakerName'
evt_col  = 'event_id'        if 'event_id'        in df.columns else 'matchId'

per_event = (df.groupby(['market', evt_col])[book_col].nunique().reset_index(name='n_bookies'))

avg_bookies = (per_event.groupby('market')['n_bookies']
                         .mean()
                         .sort_values(ascending=False))

print(avg_bookies.round(2))

plt.figure(figsize=(8,4))
sns.barplot(x=avg_bookies.index, y=avg_bookies.values)
plt.title('Среднее число букмекеров на событие по рынкам')
plt.xlabel('market'); plt.ylabel('среднее число букмекеров')
plt.show()


In [ ]:
per_event = df.groupby(['market', evt_col])[book_col].nunique().reset_index(name='n_bookies')
avg_per_market = (per_event.groupby('market')['n_bookies']
                             .mean()
                             .reset_index(name='avg_n_bookies')
                             .sort_values('avg_n_bookies', ascending=False))

fig = px.bar(
    avg_per_market,
    x='market', y='avg_n_bookies',
    text='avg_n_bookies',
    title='Среднее число уникальных букмекеров на событие по рынкам',
    labels={'market':'market', 'avg_n_bookies':'среднее число букмекеров'},
    template='plotly_white'
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(yaxis=dict(tickformat='.2f'), bargap=0.25, height=450)
fig.show()



In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

book_col = 'bookmaker_title' if 'bookmaker_title' in df.columns else 'bookmakerName'
evt_col  = 'event_id'        if 'event_id'        in df.columns else 'matchId'
name_col = 'outcome_name'    if 'outcome_name'    in df.columns else 'outcomeName'
odds_col = 'outcome_odds'    if 'outcome_odds'    in df.columns else 'outcomeOdds'
time_col = 'snapshot_time'

df[odds_col] = pd.to_numeric(df[odds_col], errors='coerce')
df[time_col] = pd.to_datetime(df[time_col], errors='coerce')

var_tbl = (
    df.groupby(['market', evt_col, name_col, book_col])
      .agg(std_odds=(odds_col, 'std'),
           n_snap=(time_col, 'nunique'))
      .reset_index()
)

var_tbl = var_tbl[var_tbl['n_snap'] >= 2].dropna(subset=['std_odds'])

mean_std_by_market = (var_tbl.groupby('market')['std_odds']
                              .mean()
                              .sort_values(ascending=True))
print(mean_std_by_market.round(4))

plt.figure(figsize=(8,4))
sns.barplot(x=mean_std_by_market.index, y=mean_std_by_market.values)
plt.title('Средний разброс коэффициентов по snapshot_time для каждого market')
plt.xlabel('market'); plt.ylabel('средний std по времени')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

time_candidates = ['snapshot_time', 'collected_time', 'collected_at', 'collected']
time_col = next((c for c in time_candidates if c in df.columns), None)
assert time_col is not None, "Не нашли колонку времени в данных"

if time_col == 'snapshot_time' and df['snapshot_time'].nunique() <= 2 and 'collected_time' in df.columns:
    time_col = 'collected_time'
elif time_col == 'snapshot_time' and df['snapshot_time'].nunique() <= 2 and 'collected_at' in df.columns:
    time_col = 'collected_at'

print("Используем временную колонку:", time_col)

df['outcome_odds']  = pd.to_numeric(df['outcome_odds'], errors='coerce')
df[time_col]        = pd.to_datetime(df[time_col], errors='coerce')

var_tbl = (
    df.groupby(['market', 'event_id', 'outcome_name', 'bookmaker_title'])
      .agg(std_odds=('outcome_odds', 'std'),
           n_snap=(time_col, 'nunique'))
      .reset_index()
)

print(var_tbl['n_snap'].value_counts().sort_index().head(10))

var_tbl = var_tbl[(var_tbl['n_snap'] >= 2) & var_tbl['std_odds'].notna()]
assert len(var_tbl) > 0, "Нет групп с ≥2 временными срезами — проверь временную колонку/данные"

plt.figure(figsize=(10,5))
sns.boxplot(data=var_tbl, x='market', y='std_odds')
plt.title('Распределение разброса коэффициентов по времени (по рынкам)')
plt.xlabel('Market');
plt.ylabel('std outcome_odds по времени')
plt.ylim(0,0.1)
plt.show()

mean_std = var_tbl.groupby('market')['std_odds'].mean().sort_values()
plt.figure(figsize=(8,4))
sns.barplot(x=mean_std.index, y=mean_std.values)
plt.title('Средний std outcome_odds по времени для каждого market')
plt.xlabel('Market');
plt.ylabel('средний std')
plt.show()


In [ ]:
df['total_goals'] = df['homeResult'] + df['awayResult']
df['goal_diff'] = df['homeResult'] - df['awayResult']

plt.figure()
sns.countplot(x=df['homeResult'])
plt.title('Столбчатая диаграмма для результатов домашних матчей')
plt.xlabel('Результат домашних матчей')
plt.ylabel('Частота')
plt.show()

plt.figure()
sns.countplot(x=df['awayResult'])
plt.title('Столбчатая диаграмма для результатов гостевых матчей')
plt.xlabel('Результат гостевых матчей')
plt.ylabel('Частота')
plt.show()

plt.figure()
sns.countplot(x=df['total_goals'])
plt.title('Столбчатая диаграмма для общего количества голов')
plt.xlabel('Общее количество голов')
plt.ylabel('Частота')
plt.show()

plt.figure(figsize=(12,6))
sns.countplot(x=df['goal_diff'])
plt.title('Столбчатая диаграмма для разницы голов')
plt.xlabel('Разница голов')
plt.ylabel('Частота')
plt.show()


In [ ]:
df['total_goals'] = df['homeResult'] + df['awayResult']
df['goal_diff'] = df['homeResult'] - df['awayResult']

fig1 = px.histogram(
    df, x='total_goals', nbins=15, color_discrete_sequence=['skyblue'],
    title='Распределение общего количества голов в матчах'
)
fig1.update_layout(xaxis_title='Общее количество голов', yaxis_title='Частота', template='plotly_white')
fig1.show()

df['score'] = df['homeResult'].astype(str) + ':' + df['awayResult'].astype(str)
score_counts = df['score'].value_counts().reset_index()
score_counts.columns = ['score', 'count']

fig2 = px.bar(
    score_counts.head(15),
    x='score', y='count', text='count',
    title='Наиболее частые счета матчей (топ-15)',
    color_discrete_sequence=['orange']
)
fig2.update_traces(textposition='outside')
fig2.update_layout(xaxis_title='Счёт', yaxis_title='Количество матчей', template='plotly_white')
fig2.show()

fig3 = px.histogram(
    df, x='goal_diff', nbins=15, color_discrete_sequence=['lightgreen'],
    title='Распределение разницы голов'
)
fig3.update_layout(xaxis_title='Разница голов (home - away)', yaxis_title='Частота', template='plotly_white')
fig3.show()


In [ ]:
for col in ['Match Winner_Home', 'Match Winner_Draw', 'Match Winner_Away']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['implied_home_prob'] = 1 / df['Match Winner_Home']
df['implied_draw_prob'] = 1 / df['Match Winner_Draw']
df['implied_away_prob'] = 1 / df['Match Winner_Away']

# 3️⃣ Фактические исходы
df['actual_home'] = (df['homeResult'] > df['awayResult']).astype(int)
df['actual_draw'] = (df['homeResult'] == df['awayResult']).astype(int)
df['actual_away'] = (df['homeResult'] < df['awayResult']).astype(int)

bins = np.arange(1.0, 10.5, 0.5)
labels = [f"[{bins[i]}–{bins[i+1]})" for i in range(len(bins)-1)]
for col in ['Match Winner_Home', 'Match Winner_Draw', 'Match Winner_Away']:
    df[f'{col}_bin'] = pd.cut(df[col], bins=bins, labels=labels, include_lowest=True)

results = []
for kind, imp_col, act_col in [
    ('Home', 'implied_home_prob', 'actual_home'),
    ('Draw', 'implied_draw_prob', 'actual_draw'),
    ('Away', 'implied_away_prob', 'actual_away')
]:
    group = df.groupby(f'Match Winner_{kind}_bin').agg(
        mean_implied_prob=(imp_col, 'mean'),
        actual_frequency=(act_col, 'mean')
    ).reset_index()
    group['diff'] = group['actual_frequency'] - group['mean_implied_prob']
    group['outcome'] = kind
    results.append(group)

final = pd.concat(results)

fig1 = px.line(
    final,
    x='Match Winner_Home_bin', y=['mean_implied_prob', 'actual_frequency'],
    color_discrete_sequence=['blue', 'orange'],
    title='Подразумеваемая и фактическая вероятность по диапазонам коэффициентов (Home)',
    labels={'value': 'Вероятность', 'Match Winner_Home_bin': 'Диапазон коэффициентов'}
)
fig1.update_layout(template='plotly_white')
fig1.show()

fig2 = px.bar(
    final, x='Match Winner_Home_bin', y='diff', color='outcome',
    title='Разница между фактической и подразумеваемой вероятностью (actual - implied)',
    labels={'diff': 'Разница', 'Match Winner_Home_bin': 'Диапазон коэффициентов'},
    template='plotly_white'
)
fig2.show()


In [ ]:
import numpy as np, pandas as pd
import plotly.express as px

for c in ['Match Winner_Home','Match Winner_Draw','Match Winner_Away']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['implied_home_prob'] = 1/df['Match Winner_Home']
df['implied_draw_prob'] = 1/df['Match Winner_Draw']
df['implied_away_prob'] = 1/df['Match Winner_Away']
df['actual_home'] = (df['homeResult'] > df['awayResult']).astype(int)
df['actual_draw'] = (df['homeResult'] == df['awayResult']).astype(int)
df['actual_away'] = (df['homeResult'] < df['awayResult']).astype(int)

bins = np.arange(1.0, 10.5, 0.5)
labels = [f"[{bins[i]}–{bins[i+1]})" for i in range(len(bins)-1)]

def prepare(odds_col, imp_col, act_col, outcome_name):
    tmp = df[[odds_col, imp_col, act_col]].dropna().copy()
    tmp['bin'] = pd.cut(tmp[odds_col], bins=bins, labels=labels, include_lowest=True, right=False)
    g = (tmp.groupby('bin', observed=True)
            .agg(mean_implied=(imp_col,'mean'), actual=(act_col,'mean'))
            .reset_index())
    g['diff'] = g['actual'] - g['mean_implied']
    g['outcome'] = outcome_name
    return g

g_home = prepare('Match Winner_Home','implied_home_prob','actual_home','Победа хозяев')
g_draw = prepare('Match Winner_Draw','implied_draw_prob','actual_draw','Ничья')
g_away = prepare('Match Winner_Away','implied_away_prob','actual_away','Победа гостей')

def lines_plot(g, title):
    long = g.melt(id_vars=['bin','outcome'], value_vars=['mean_implied','actual'],
                  var_name='metric', value_name='prob')
    fig = px.line(long, x='bin', y='prob', color='metric',
                  category_orders={'bin': labels, 'metric':['mean_implied','actual']},
                  labels={'bin':'Диапазон коэффициентов','prob':'Вероятность','metric':'Метрика'},
                  title=title, template='plotly_white')
    fig.update_layout(xaxis_tickangle=45, height=480, legend_title_text='')
    fig.show()

def diff_bar(g, title):
    fig = px.bar(g, x='bin', y='diff', color='diff',
                 category_orders={'bin': labels},
                 color_continuous_scale='RdBu_r', range_color=[-0.25,0.25],
                 labels={'bin':'Диапазон коэффициентов','diff':'Разница (actual − implied)'},
                 title=title, template='plotly_white')
    fig.add_hline(y=0, line_dash='dot', line_color='gray')
    fig.update_layout(xaxis_tickangle=45, height=480, coloraxis_colorbar_title='diff')
    fig.show()

lines_plot(g_home, 'Implied vs Actual по бинам: Победа хозяев')
diff_bar (g_home, 'Разница (actual − implied) по бинам: Победа хозяев')

lines_plot(g_draw, 'Implied vs Actual по бинам: Ничья')
diff_bar (g_draw, 'Разница (actual − implied) по бинам: Ничья')

lines_plot(g_away, 'Implied vs Actual по бинам: Победа гостей')
diff_bar (g_away, 'Разница (actual − implied) по бинам: Победа гостей')


In [ ]:
for c in ['Match Winner_Home','Match Winner_Draw','Match Winner_Away']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['implied_home_prob'] = 1 / df['Match Winner_Home']
df['implied_draw_prob'] = 1 / df['Match Winner_Draw']
df['implied_away_prob'] = 1 / df['Match Winner_Away']

df['actual_home'] = (df['homeResult'] > df['awayResult']).astype(int)
df['actual_draw'] = (df['homeResult'] == df['awayResult']).astype(int)
df['actual_away'] = (df['homeResult'] < df['awayResult']).astype(int)

bins = np.arange(1.0, 10.5, 0.5)
labels = [f"[{bins[i]}–{bins[i+1]})" for i in range(len(bins)-1)]

def prep(kind, odds_col, imp_col, act_col):
    tmp = df[[odds_col, imp_col, act_col]].dropna().copy()
    tmp['bin'] = pd.cut(tmp[odds_col], bins=bins, labels=labels, include_lowest=True, right=False)
    g = (tmp.groupby('bin', observed=True)
            .agg(mean_implied_prob=(imp_col, 'mean'),
                 actual_frequency=(act_col, 'mean'))
            .reset_index())
    g['outcome'] = kind
    return g

home = prep('Победа хозяев', 'Match Winner_Home', 'implied_home_prob', 'actual_home')
draw = prep('Ничья',         'Match Winner_Draw', 'implied_draw_prob', 'actual_draw')
away = prep('Победа гостей', 'Match Winner_Away', 'implied_away_prob', 'actual_away')

def plot_lines(df_plot, title):
    long = df_plot.melt(id_vars=['bin'],
                        value_vars=['mean_implied_prob','actual_frequency'],
                        var_name='Метрика', value_name='Вероятность')
    fig = px.line(
        long, x='bin', y='Вероятность', color='Метрика',
        markers=True,
        category_orders={'bin': labels, 'Метрика':['mean_implied_prob','actual_frequency']},
        labels={'bin':'Диапазон коэффициентов'},
        title=title, template='plotly_white'
    )
    fig.update_layout(xaxis_tickangle=45, legend_title_text='', height=450)
    fig.show()

plot_lines(home, 'Implied vs Actual: Победа хозяев')
plot_lines(draw, 'Implied vs Actual: Ничья')
plot_lines(away, 'Implied vs Actual: Победа гостей')


In [ ]:
for c in ['Match Winner_Home','Match Winner_Draw','Match Winner_Away']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['implied_home_prob'] = 1/df['Match Winner_Home']
df['implied_draw_prob'] = 1/df['Match Winner_Draw']
df['implied_away_prob'] = 1/df['Match Winner_Away']
df['actual_home'] = (pd.to_numeric(df['homeResult'],errors='coerce') >  pd.to_numeric(df['awayResult'],errors='coerce')).astype(int)
df['actual_draw'] = (pd.to_numeric(df['homeResult'],errors='coerce') == pd.to_numeric(df['awayResult'],errors='coerce')).astype(int)
df['actual_away'] = (pd.to_numeric(df['homeResult'],errors='coerce') <  pd.to_numeric(df['awayResult'],errors='coerce')).astype(int)

bins = np.arange(1.0, 10.5, 0.5)
labels = [f"[{bins[i]}–{bins[i+1]})" for i in range(len(bins)-1)]

def prep(odds_col, imp_col, act_col):
    t = df[[odds_col, imp_col, act_col]].dropna().copy()
    t['bin'] = pd.cut(t[odds_col], bins=bins, labels=labels, include_lowest=True, right=False)
    g = (t.groupby('bin', observed=True)
           .agg(mean_implied_prob=(imp_col,'mean'),
                actual_frequency=(act_col,'mean'))
           .reset_index())
    g['diff'] = g['actual_frequency'] - g['mean_implied_prob']
    return g

g_home = prep('Match Winner_Home','implied_home_prob','actual_home')
g_draw = prep('Match Winner_Draw','implied_draw_prob','actual_draw')
g_away = prep('Match Winner_Away','implied_away_prob','actual_away')

def plot_diff(g, title):
    fig = px.bar(g, x='bin', y='diff', color='diff',
                 category_orders={'bin': labels},
                 color_continuous_scale='RdBu_r', range_color=[-0.25,0.25],
                 labels={'bin':'Диапазон коэффициентов','diff':'actual − implied'},
                 title=title, template='plotly_white')
    fig.add_hline(y=0, line_dash='dot', line_color='gray')
    fig.update_layout(xaxis_tickangle=45, height=480, coloraxis_colorbar_title='diff')
    fig.show()

plot_diff(g_home, 'Разница (actual − implied) по бинам: Победа хозяев')
plot_diff(g_draw, 'Разница (actual − implied) по бинам: Ничья')
plot_diff(g_away, 'Разница (actual − implied) по бинам: Победа гостей')


In [ ]:

for c in ['Match Winner_Home','Match Winner_Draw','Match Winner_Away']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['implied_home'] = 1 / df['Match Winner_Home']
df['implied_draw'] = 1 / df['Match Winner_Draw']
df['implied_away'] = 1 / df['Match Winner_Away']

df['actual_home'] = (df['homeResult'] > df['awayResult']).astype(int)
df['actual_draw'] = (df['homeResult'] == df['awayResult']).astype(int)
df['actual_away'] = (df['homeResult'] < df['awayResult']).astype(int)

df['diff_home'] = df['actual_home'] - df['implied_home']
df['diff_draw'] = df['actual_draw'] - df['implied_draw']
df['diff_away'] = df['actual_away'] - df['implied_away']

league_diff = (
    df.groupby('leagueName')[['diff_home','diff_draw','diff_away']]
      .mean()
      .sort_values('diff_home', ascending=False)
      .reset_index()
)

df['country_pair'] = df['homeTeamCountry'] + ' vs ' + df['awayTeamCountry']
country_diff = (
    df.groupby('country_pair')[['diff_home','diff_draw','diff_away']]
      .mean()
      .sort_values('diff_home', ascending=False)
      .reset_index()
)

#визуализация по лигам
fig1 = px.bar(
    league_diff.head(15).melt(id_vars='leagueName', var_name='Тип исхода', value_name='Разница'),
    x='leagueName', y='Разница', color='Тип исхода',
    title='Средняя разница (actual − implied) по лигам',
    template='plotly_white'
)
fig1.update_layout(xaxis_tickangle=45, height=500)
fig1.add_hline(y=0, line_dash='dot', line_color='gray')
fig1.show()

# визуализация по странам
fig2 = px.bar(
    country_diff.head(15).melt(id_vars='country_pair', var_name='Тип исхода', value_name='Разница'),
    x='country_pair', y='Разница', color='Тип исхода',
    title='Средняя разница (actual − implied) по странам (home vs away)',
    template='plotly_white'
)
fig2.update_layout(xaxis_tickangle=45, height=500)
fig2.add_hline(y=0, line_dash='dot', line_color='gray')
fig2.show()


In [ ]:
for c in ['Match Winner_Home','Match Winner_Draw','Match Winner_Away']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['implied_home_prob'] = 1 / df['Match Winner_Home']
df['implied_draw_prob'] = 1 / df['Match Winner_Draw']
df['implied_away_prob'] = 1 / df['Match Winner_Away']

df['actual_home'] = (pd.to_numeric(df['homeResult'], errors='coerce') >  pd.to_numeric(df['awayResult'], errors='coerce')).astype(int)
df['actual_draw'] = (pd.to_numeric(df['homeResult'], errors='coerce') == pd.to_numeric(df['awayResult'], errors='coerce')).astype(int)
df['actual_away'] = (pd.to_numeric(df['homeResult'], errors='coerce') <  pd.to_numeric(df['awayResult'], errors='coerce')).astype(int)

df['err_home'] = (df['implied_home_prob'] - df['actual_home']).abs()
df['err_draw'] = (df['implied_draw_prob'] - df['actual_draw']).abs()
df['err_away'] = (df['implied_away_prob'] - df['actual_away']).abs()
df['err_mean'] = df[['err_home','err_draw','err_away']].mean(axis=1)

err_by_bk_league = (
    df.groupby(['bookmakerName','leagueName'])
      .agg(n=('err_mean','size'),
           err_home_mean=('err_home','mean'),
           err_draw_mean=('err_draw','mean'),
           err_away_mean=('err_away','mean'),
           err_overall_mean=('err_mean','mean'))
      .reset_index()
      .sort_values(['leagueName','err_overall_mean'])
)

err_by_bk_country = (
    df.groupby(['bookmakerName','homeTeamCountry'])
      .agg(n=('err_mean','size'),
           err_home_mean=('err_home','mean'),
           err_draw_mean=('err_draw','mean'),
           err_away_mean=('err_away','mean'),
           err_overall_mean=('err_mean','mean'))
      .reset_index()
      .sort_values(['homeTeamCountry','err_overall_mean'])
)

print('Ошибки по букмекеру в лиге (первые строки):')
print(err_by_bk_league.head(20), '\n')

print('Ошибки по букмекеру в стране (первые строки):')
print(err_by_bk_country.head(20))


In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

for c in ['Match Winner_Home','Match Winner_Draw','Match Winner_Away']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['implied_home_prob'] = 1 / df['Match Winner_Home']
df['implied_draw_prob'] = 1 / df['Match Winner_Draw']
df['implied_away_prob'] = 1 / df['Match Winner_Away']

df['actual_home'] = (df['homeResult'] > df['awayResult']).astype(int)
df['actual_draw'] = (df['homeResult'] == df['awayResult']).astype(int)
df['actual_away'] = (df['homeResult'] < df['awayResult']).astype(int)

df['err_home'] = (df['implied_home_prob'] - df['actual_home']).abs()
df['err_draw'] = (df['implied_draw_prob'] - df['actual_draw']).abs()
df['err_away'] = (df['implied_away_prob'] - df['actual_away']).abs()
df['err_mean'] = df[['err_home','err_draw','err_away']].mean(axis=1)

fig1 = px.box(df, x='leagueName', y='err_mean',
              points='outliers',
              title='Распределение ошибки букмекеров по лигам',
              labels={'leagueName':'Лига','err_mean':'Ошибка букмекера'},
              template='plotly_white')
fig1.update_layout(xaxis_tickangle=45, height=500)
fig1.show()

fig2 = px.box(df, x='bookmakerName', y='err_mean',
              points='outliers',
              title='Распределение ошибки букмекеров по букмекерам',
              labels={'bookmakerName':'Букмекер','err_mean':'Ошибка букмекера'},
              template='plotly_white')
fig2.update_layout(xaxis_tickangle=45, height=500)
fig2.show()
